# Local Model Chat Test

This notebook is a small playground for testing local model chat behavior.

It lets you:
- load a local Hugging Face model with the existing runtime
- run a quick single-prompt smoke test
- ask repeated questions from a notebook text box


> This notebook is configured for a Gemma 4 text-only community conversion on Windows, so it uses the plain tokenizer path instead of the multimodal Gemma 4 processor path.


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "local_retrieval_model.py").exists():
    candidate_dir = NOTEBOOK_DIR / "src" / "llm_rl_playground"
    if candidate_dir.exists():
        NOTEBOOK_DIR = candidate_dir

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print(f"Using notebook module path: {NOTEBOOK_DIR}")


Using notebook module path: c:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\src\llm_rl_playground


In [2]:
import torch
from IPython.display import Markdown, display, clear_output
import ipywidgets as widgets

from local_retrieval_model import load_local_retrieval_model


In [3]:
# Change these values for your model.
# Default: Gemma 4 text-only community conversion for Windows/local testing.
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct" #"google/gemma-2-2b-it" "principled-intelligence/gemma-4-E2B-it-text-only"
ADAPTER_PATH = None
USE_4BIT = True
ENABLE_THINKING = False

print({
    "MODEL_ID": MODEL_ID,
    "ADAPTER_PATH": ADAPTER_PATH,
    "USE_4BIT": USE_4BIT,
    "ENABLE_THINKING": ENABLE_THINKING,
    "cuda_available": torch.cuda.is_available(),
})


{'MODEL_ID': 'Qwen/Qwen2.5-1.5B-Instruct', 'ADAPTER_PATH': None, 'USE_4BIT': True, 'ENABLE_THINKING': False, 'cuda_available': True}


In [4]:
runtime = load_local_retrieval_model(
    model_id=MODEL_ID,
    adapter_path=ADAPTER_PATH,
    use_4bit=USE_4BIT,
    enable_thinking=ENABLE_THINKING,
)

print(type(runtime).__name__)
print(f"Loaded model: {runtime.model_id}")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

c:\Users\jmgil\Desktop\TESE\LATTICE\.venvLattice\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jmgil\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

c:\Users\jmgil\Desktop\TESE\LATTICE\.venvLattice\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LocalRetrievalModelRuntime
Loaded model: Qwen/Qwen2.5-1.5B-Instruct


In [5]:
def ask_local_model(question, system_prompt=None, max_new_tokens=256, temperature=0.2, top_p=0.95, enable_thinking=None):
    return runtime.ask(
        question=question,
        system_prompt=system_prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        enable_thinking=enable_thinking,
    )


In [6]:
TEST_PROMPT = "Explain in 3 short bullet points what this model can help me test locally."
print(ask_local_model(TEST_PROMPT))


c:\Users\jmgil\Desktop\TESE\LATTICE\.venvLattice\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


- Run code snippets directly in your browser without needing an IDE.
- Quickly experiment with different configurations and settings for testing purposes.
- Easily integrate with other development tools like Git or Docker to manage dependencies.


## Notebook Chat Box

Type a question and click **Generate**. Each request is independent unless you put extra context in the system prompt.


In [ ]:
system_prompt_box = widgets.Textarea(
    value="",
    placeholder="Optional system prompt...",
    description="System:",
    layout=widgets.Layout(width="100%", height="90px"),
)

prompt_box = widgets.Textarea(
    value="",
    placeholder="Ask the local model a question...",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="140px"),
)

settings_box = widgets.HBox([
    widgets.IntSlider(value=256, min=32, max=1024, step=32, description="Tokens"),
    widgets.FloatSlider(value=0.2, min=0.0, max=1.5, step=0.1, readout_format='.1f', description="Temp"),
    widgets.FloatSlider(value=0.95, min=0.1, max=1.0, step=0.05, readout_format='.2f', description="Top-p"),
])

generate_button = widgets.Button(description="Generate", button_style="primary")
clear_button = widgets.Button(description="Clear Output")
output_box = widgets.Output()

token_slider = settings_box.children[0]
temperature_slider = settings_box.children[1]
top_p_slider = settings_box.children[2]

def on_generate_clicked(_):
    question = prompt_box.value.strip()
    system_prompt = system_prompt_box.value.strip() or None

    with output_box:
        clear_output(wait=True)
        if not question:
            print("Please write a prompt first.")
            return

        print("Generating...")
        reply = ask_local_model(
            question,
            system_prompt=system_prompt,
            max_new_tokens=token_slider.value,
            temperature=temperature_slider.value,
            top_p=top_p_slider.value,
        )
        clear_output(wait=True)
        display(Markdown(f"**You**\n\n{question}"))
        if system_prompt:
            display(Markdown(f"**System Prompt**\n\n{system_prompt}"))
        display(Markdown(f"**Model**\n\n{reply}"))

def on_clear_clicked(_):
    with output_box:
        clear_output(wait=True)

generate_button.on_click(on_generate_clicked)
clear_button.on_click(on_clear_clicked)

display(system_prompt_box)
display(prompt_box)
display(settings_box)
display(widgets.HBox([generate_button, clear_button]))
display(output_box)


In [7]:
# Run this cell when you are done with the model.
runtime.unload()
print("Local model unloaded.")


Local model unloaded.
